In [1]:
## library imports
import numpy as np 
import matplotlib as mpl
import matplotlib.pyplot as plt 
from astropy.io import fits
from astropy.table import Table, unique
import astropy.units as u 
from astropy.cosmology import WMAP9 as cosmo
import glob
import os
import astro_functions
from halpha_fit import narrow_fit_ha, broad_fit_ha

## global constants 

ha_line = 0.6564614 # microns
hb_line = 0.4862721 # microns
o3_line = 0.5008239 # microns 

g235h_wv_range = np.array([1.66, 3.17])
g395h_wv_range = np.array([2.87, 5.27])

plt.rcParams.update({
"text.usetex": False,
'axes.labelsize' : 16,
'xtick.labelsize' : 14,
'ytick.labelsize' : 14,
'xtick.direction' : "in",
'ytick.direction' : "in",
'xtick.top' : True,
'ytick.right' : True,
"xtick.minor.visible" : False,
"ytick.minor.visible" : False,
"font.family" : "serif"})

In [2]:
## redshift catalog for sources 
oceans_redshifts = Table.read("/Users/madisynbrooks/Documents/OCEANS/OCEANS_CEERS_v1.0cat_DB_catalog_full.fits")
oceans_redshifts.colnames

['Cox_v10_number',
 'zspec',
 'oceans_id',
 'logM_50',
 'logM_16',
 'logM_84',
 'logSFRinst_50',
 'logSFRinst_16',
 'logSFRinst_84',
 'logZsol_50',
 'logZsol_16',
 'logZsol_84',
 'Av_50',
 'Av_16',
 'Av_84',
 'zfit_50',
 'zfit_16',
 'zfit_84',
 'logMt_50',
 'logMt_16',
 'logMt_84',
 'logSFRtuple_50',
 'logSFRtuple_16',
 'logSFRtuple_84',
 'logSFR10_50',
 'logSFR10_16',
 'logSFR10_84',
 'logSFR100_50',
 'logSFR100_16',
 'logSFR100_84',
 'logSFR300_50',
 'logSFR300_16',
 'logSFR300_84',
 'logSFR1000_50',
 'logSFR1000_16',
 'logSFR1000_84',
 't25_50',
 't25_16',
 't25_84',
 't50_50',
 't50_16',
 't50_84',
 't75_50',
 't75_16',
 't75_84',
 'sfh_50',
 'sfh_16',
 'sfh_84',
 'timeax',
 'obs_sed',
 'obs_err',
 'bf_sed',
 'sed_rf',
 'bf_lam',
 'bf_spec',
 'posterior_sample_ids',
 'nparam',
 'nbands',
 'chi2',
 'fit_flags']

In [3]:
len(oceans_redshifts)

205

In [4]:
## i know the redshift for source 20504 is wrong, so I am going to manually update it in the catalog 
row = oceans_redshifts['oceans_id'] == 20504

In [5]:
oceans_redshifts[row]

Cox_v10_number,zspec,oceans_id,logM_50,logM_16,logM_84,logSFRinst_50,logSFRinst_16,logSFRinst_84,logZsol_50,logZsol_16,logZsol_84,Av_50,Av_16,Av_84,zfit_50,zfit_16,zfit_84,logMt_50,logMt_16,logMt_84,logSFRtuple_50,logSFRtuple_16,logSFRtuple_84,logSFR10_50,logSFR10_16,logSFR10_84,logSFR100_50,logSFR100_16,logSFR100_84,logSFR300_50,logSFR300_16,logSFR300_84,logSFR1000_50,logSFR1000_16,logSFR1000_84,t25_50,t25_16,t25_84,t50_50,t50_16,t50_84,t75_50,t75_16,t75_84,sfh_50,sfh_16,sfh_84,timeax,obs_sed,obs_err,bf_sed,sed_rf,bf_lam,bf_spec,posterior_sample_ids,nparam,nbands,chi2,fit_flags
int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64[100],float64[100],float64[100],float64[100],float64[14],float64[14],float64[14],float64[20],float64[5994],float64[5994],float64[1000],float64,float64,float64,float64
26478,7.2306,20504.0,10.574500000002196,--,10.574500000002196,3.0325000000030164,3.0325000000030164,3.1225000000030465,-0.7255000000000853,-0.7255000000000853,-0.5515000000001045,2.4445,2.4445,2.4615,7.200523943290917,7.200523943290917,7.271523943290917,10.629500000002215,--,10.629500000002215,3.0325000000030164,3.0325000000030164,3.121500000003047,3.022416416099506,3.022416416099506,3.0841496664334858,2.63465669912417,--,2.63465669912417,2.1571628156065485,--,2.1571628156065485,1.7808646394593655,--,1.7808646394593655,0.3665,0.3665,0.9515,0.5135,0.5135,0.9724999999999999,0.9205,0.9205,0.9874999999999999,5.742657790966485 .. 901.6222769245467,3.6165684809447565 .. 901.6222769245467,5.742657790966485 .. 1183.879342442719,0.0 .. 0.7234832285559532,0.042969142913818356 .. --,0.0038201920135162897 .. --,0.026505792374560373 .. 1.698935904927846e-36,33.7401034548302 .. 2065.219261008019,0.07527184315746961 .. 82716.31116205451,2.3581381780449795e-65 .. 5453.996784243624,972619.0 .. 972619.0,3.0,11.0,32.45129443086762,0.0
26478,7.2307,20504.0,10.574500000002196,--,10.574500000002196,3.0325000000030164,3.0325000000030164,3.1225000000030465,-0.7255000000000853,-0.7255000000000853,-0.5515000000001045,2.4445,2.4445,2.4615,7.200523943290917,7.200523943290917,7.271523943290917,10.629500000002215,--,10.629500000002215,3.0325000000030164,3.0325000000030164,3.121500000003047,3.022416416099506,3.022416416099506,3.0841496664334858,2.63465669912417,--,2.63465669912417,2.1571628156065485,--,2.1571628156065485,1.7808646394593655,--,1.7808646394593655,0.3665,0.3665,0.9515,0.5135,0.5135,0.9724999999999999,0.9205,0.9205,0.9874999999999999,5.742657790966485 .. 901.6222769245467,3.6165684809447565 .. 901.6222769245467,5.742657790966485 .. 1183.879342442719,0.0 .. 0.7234832285559532,0.042969142913818356 .. --,0.0038201920135162897 .. --,0.026505792374560373 .. 1.698935904927846e-36,33.7401034548302 .. 2065.219261008019,0.07527184315746961 .. 82716.31116205451,2.3581381780449795e-65 .. 5453.996784243624,972619.0 .. 972619.0,3.0,11.0,32.45129443086762,0.0


In [6]:
oceans_redshifts['zspec'][row] = 5.276

In [7]:
oceans_redshifts[row]

Cox_v10_number,zspec,oceans_id,logM_50,logM_16,logM_84,logSFRinst_50,logSFRinst_16,logSFRinst_84,logZsol_50,logZsol_16,logZsol_84,Av_50,Av_16,Av_84,zfit_50,zfit_16,zfit_84,logMt_50,logMt_16,logMt_84,logSFRtuple_50,logSFRtuple_16,logSFRtuple_84,logSFR10_50,logSFR10_16,logSFR10_84,logSFR100_50,logSFR100_16,logSFR100_84,logSFR300_50,logSFR300_16,logSFR300_84,logSFR1000_50,logSFR1000_16,logSFR1000_84,t25_50,t25_16,t25_84,t50_50,t50_16,t50_84,t75_50,t75_16,t75_84,sfh_50,sfh_16,sfh_84,timeax,obs_sed,obs_err,bf_sed,sed_rf,bf_lam,bf_spec,posterior_sample_ids,nparam,nbands,chi2,fit_flags
int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64[100],float64[100],float64[100],float64[100],float64[14],float64[14],float64[14],float64[20],float64[5994],float64[5994],float64[1000],float64,float64,float64,float64
26478,5.276,20504.0,10.574500000002196,--,10.574500000002196,3.0325000000030164,3.0325000000030164,3.1225000000030465,-0.7255000000000853,-0.7255000000000853,-0.5515000000001045,2.4445,2.4445,2.4615,7.200523943290917,7.200523943290917,7.271523943290917,10.629500000002215,--,10.629500000002215,3.0325000000030164,3.0325000000030164,3.121500000003047,3.022416416099506,3.022416416099506,3.0841496664334858,2.63465669912417,--,2.63465669912417,2.1571628156065485,--,2.1571628156065485,1.7808646394593655,--,1.7808646394593655,0.3665,0.3665,0.9515,0.5135,0.5135,0.9724999999999999,0.9205,0.9205,0.9874999999999999,5.742657790966485 .. 901.6222769245467,3.6165684809447565 .. 901.6222769245467,5.742657790966485 .. 1183.879342442719,0.0 .. 0.7234832285559532,0.042969142913818356 .. --,0.0038201920135162897 .. --,0.026505792374560373 .. 1.698935904927846e-36,33.7401034548302 .. 2065.219261008019,0.07527184315746961 .. 82716.31116205451,2.3581381780449795e-65 .. 5453.996784243624,972619.0 .. 972619.0,3.0,11.0,32.45129443086762,0.0
26478,5.276,20504.0,10.574500000002196,--,10.574500000002196,3.0325000000030164,3.0325000000030164,3.1225000000030465,-0.7255000000000853,-0.7255000000000853,-0.5515000000001045,2.4445,2.4445,2.4615,7.200523943290917,7.200523943290917,7.271523943290917,10.629500000002215,--,10.629500000002215,3.0325000000030164,3.0325000000030164,3.121500000003047,3.022416416099506,3.022416416099506,3.0841496664334858,2.63465669912417,--,2.63465669912417,2.1571628156065485,--,2.1571628156065485,1.7808646394593655,--,1.7808646394593655,0.3665,0.3665,0.9515,0.5135,0.5135,0.9724999999999999,0.9205,0.9205,0.9874999999999999,5.742657790966485 .. 901.6222769245467,3.6165684809447565 .. 901.6222769245467,5.742657790966485 .. 1183.879342442719,0.0 .. 0.7234832285559532,0.042969142913818356 .. --,0.0038201920135162897 .. --,0.026505792374560373 .. 1.698935904927846e-36,33.7401034548302 .. 2065.219261008019,0.07527184315746961 .. 82716.31116205451,2.3581381780449795e-65 .. 5453.996784243624,972619.0 .. 972619.0,3.0,11.0,32.45129443086762,0.0


In [8]:
oceans_redshifts = oceans_redshifts['oceans_id', 'zspec']


In [9]:
## removing duplicates from the redshift catalog, sources were observed more than one time 
oceans_redshifts = unique(oceans_redshifts, keys = 'oceans_id')

In [10]:
oceans_redshifts

oceans_id,zspec
float64,float64
462.0,4.1427
597.0,3.151
1459.0,4.9465
1613.0,4.2807
1781.0,4.3493
1785.0,8.3299
1794.0,3.6817
1994.0,5.4228
2255.0,2.9669


In [11]:
## loop through the catalog and find the path to the file for each source 
data_direc = "/Users/madisynbrooks/Documents/OCEANS/Data/v0.5/x1d"
background = "mb" ## set backgroun to mb for master or nodded for local 

ha_direc = '/Users/madisynbrooks/Documents/OCEANS/Emission Line Fitting/halpha fits'

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm 

## prevent thread oversubscription
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

fit_type = "narrow"  # tells us which model to run 

def process_single_source(row, data_direc, ha_direc, background, ha_line, g235h_wv_range, g395h_wv_range, fit_type):
    """
    Wrapper function for running mcmc in parallel. 

    Paraemters:
    ------------------
    row: 
        astropy table row of specific source 
    data_direc: str
        file path to saved spectra 
    ha_direc: str
        file path to where fits will be saved 
    background: str 
        identifies the type of background reduction used in the data processing, either "mb" or "nodded"
    ha_line: float 
        rest-frame halpha wavelength 
    g235h_wv_range: np.array
        observed wavelength range of NIRSpec 235H grating 
    g395h_wv_range: np.array
        observed wavelength range of NIRSpec 395H grating 
    fit_type: str
        identifies what model is ran for fitting emission line, currently "narrow" or "broad"

    """

    oceans_id = str(int(row['oceans_id'])).zfill(9)  # source ids are 9 numbers long with leading zeros
    z = row['zspec']
    spectra_name = f"s{oceans_id}"

    search_pattern = os.path.join(data_direc, "**", f"s{oceans_id}_{background}_*.fits")
    file_path = glob.glob(search_pattern, recursive=True)

    if len(file_path) == 0:
        print(f"No files found for source {oceans_id}")
        return

    ## check if there is more than one file/pointing per source 
    if len(file_path) > 1:
        print(f"Source {oceans_id} found in more than one pointing ({len(file_path)} files).")
        
        for file in file_path:
            full_dir = os.path.dirname(file)
            subdir_name = os.path.basename(full_dir)
            current_spectra_name = f"{subdir_name}_{spectra_name}"
            
            wv, fnu, fnu_err, flam, flam_err = astro_functions.load_1d_spec(file)
            wv_obs = wv / (1 + z)  
            
            line_obs = (1 + z) * ha_line
            if g235h_wv_range[0] < line_obs < g395h_wv_range[1]:
                if fit_type == "narrow":
                    # Make sure progress=False is set inside your MCMC run so emcee doesn't print inner bars
                    narrow_fit_ha(wv, flam, flam_err, z, current_spectra_name, ha_direc, plot_spec=True)
                elif fit_type == "broad":
                    broad_fit_ha(wv, flam, flam_err, z, current_spectra_name, ha_direc, plot_spec=True)
            else:
                print(f"No H-alpha wavelength coverage for {current_spectra_name}... ")

    else:
        wv, fnu, fnu_err, flam, flam_err = astro_functions.load_1d_spec(file_path[0])
        wv_obs = wv / (1 + z)  
        
        line_obs = (1 + z) * ha_line
        if g235h_wv_range[0] < line_obs < g395h_wv_range[1]:
            if fit_type == "narrow":
                narrow_fit_ha(wv, flam, flam_err, z, spectra_name, ha_direc, plot_spec=True)
            elif fit_type == "broad":
                broad_fit_ha(wv, flam, flam_err, z, spectra_name, ha_direc, plot_spec=True)
        else:
            print(f"No H-alpha wavelength coverage for source {oceans_id}... ")

## run the loop in parallel
print("Starting parallel processing of sources...")
Parallel(n_jobs=-1)(
    delayed(process_single_source)(
        row, data_direc, ha_direc, background, ha_line, g235h_wv_range, g395h_wv_range, fit_type
    ) 
    for row in tqdm(oceans_redshifts, desc="Fitting sources")
)
print("All sources processed!")

Starting parallel processing of sources...


Fitting sources:   8%|▊         | 16/197 [00:03<00:43,  4.20it/s]

No H-alpha wavelength coverage for source 000001785... 
fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
plotting fit...
saving halpha fit...
fitting the halpha line...
---------------------------
plotting fit...
saving halpha fit...
fitting the halpha line...
---------------------------
plotting fit...
plotting fit...
plotting fit...
plotting fit...
plotting fit...
saving halpha fit...
plotting fit...
saving halpha fit...
saving halpha fit...
saving halpha fit...


Fitting sources:  12%|█▏        | 24/197 [00:27<04:01,  1.40s/it]

fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
Velocity array is empty or flux is all NaNs. Returning NaNs.
fitting the halpha line...
---------------------------
saving halpha fit...
fitting the halpha line...
---------------------------
saving halpha fit...
fitting the halpha line...
---------------------------
fitting the halpha line...
---------------------------
plotting fit...
saving halpha fit...
fitting the halpha line...
---------------------------
Velocity array is empty or flux is all NaNs. Returning NaNs.
fitting the halpha line...
---------------------------
plotting fit...
saving halpha fit...
fitting the halpha line...
---------------------------
Velocity array is empty or flux is all NaNs. Returning NaNs.
fitting the halpha line...
---------------------------
Velocity array is empty or flux is all NaNs. Returning NaNs.
fitting the halpha line...
--------

Fitting sources:  16%|█▌        | 32/197 [00:52<05:36,  2.04s/it]

... (51 outputs left out)

/Users/madisynbrooks/Documents/OCEANS/Emission Line Fitting/halpha_fit.py:142: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize = [8,5])


saving halpha fit...
plotting fit...


/Users/madisynbrooks/Documents/OCEANS/Emission Line Fitting/halpha_fit.py:142: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize = [8,5])


saving halpha fit...
plotting fit...
saving halpha fit...
plotting fit...
saving halpha fit...
plotting fit...
saving halpha fit...
fitting the halpha line...
---------------------------
plotting fit...
saving halpha fit...
plotting fit...
saving halpha fit...
plotting fit...
saving halpha fit...
All sources processed!


In [ ]:
scale_factor = 1e18
new_rows = []

for row in oceans_redshifts:
    oceans_id = str(int(row['oceans_id'])).zfill(9)  # source ids are 9 numbers long with leading zeros
    z = row['zspec']
    spectra_name = f"s{oceans_id}"


    search_pattern = os.path.join(ha_direc, f"*{spectra_name}_narrow_ha_fit.csv")
    file_path = glob.glob(search_pattern)

    if len(file_path) == 0:
        print(f"No file found for {spectra_name}")
        continue

    for file in file_path:
        new_row = dict(row)
        
        ## open up fit parameters 
        fit_params = Table.read(file, fill_values=[('', 'nan')]) ## need to handle missing values as nans due to file saving 

        center = fit_params['Center']
        nar_amp = fit_params['Narrow Amp'] / scale_factor
        nar_sigma = fit_params['Narrow Sigma']
        cont = fit_params['Continuum'] / scale_factor

        lambda_use = (1 + z) * (ha_line * 1e5)

        n_flux = np.sqrt(2 * np.pi) * nar_amp * ((nar_sigma / 3e5) * lambda_use) * u.erg / u.s / u.cm**2
        narrow_flux = [np.percentile(n_flux.value, 16), np.percentile(n_flux.value, 50), np.percentile(n_flux.value, 84)]

        dist = (cosmo.luminosity_distance(z).to(u.cm))
        lum = (n_flux * 4 * np.pi * (dist**2)).to(u.erg / u.s)
        narrow_luminosity = np.array([np.percentile(lum.value, 16), np.percentile(lum.value, 50), np.percentile(lum.value, 84)])

        new_row['narrow_ha_flux'] = narrow_flux
        new_row['narrow_ha_luminosity'] = narrow_luminosity

        low = narrow_flux[1] - narrow_flux[0]
        high = narrow_flux[2] - narrow_flux[1]
        noise_denom = (low + high) / 2

        ##  check if the noise is effectively zero or negative to prevent division by zero / infinity
        if noise_denom > 1e-20:  # Adjust this small epsilon threshold if needed
            new_row['narrow_ha_SN'] = narrow_flux[1] / noise_denom
        else:
            new_row['narrow_ha_SN'] = np.nan  # Or flag as unconstrained/failed MCMC convergence
        # Append the populated row to our list
        new_rows.append(new_row)

## creating a new table 
oceans_redshifts_expanded = Table(new_rows)

No file found for s000001785
No file found for s000013636
No file found for s000035864
No file found for s000046768
No file found for s000053762
No file found for s000053854
No file found for s000102860
No file found for s000105404
No file found for s000132839
No file found for s000169018


In [ ]:
oceans_redshifts_expanded[np.isnan(oceans_redshifts_expanded['narrow_ha_flux'])]

oceans_id,zspec,narrow_ha_flux,narrow_ha_luminosity,narrow_ha_SN
float64,float64,float64[3],float64[3],float64
4402.0,6.6784,nan .. nan,nan .. nan,nan
6270.0,3.5964,nan .. nan,nan .. nan,nan
8860.0,5.617,nan .. nan,nan .. nan,nan
9653.0,6.5738,nan .. nan,nan .. nan,nan
10492.0,3.2221,nan .. nan,nan .. nan,nan
32111.0,5.8233,nan .. nan,nan .. nan,nan
32492.0,6.0008,nan .. nan,nan .. nan,nan
33961.0,6.9702,nan .. nan,nan .. nan,nan
34891.0,5.6724,nan .. nan,nan .. nan,nan


In [51]:
oceans_redshifts_expanded[oceans_redshifts_expanded['narrow_ha_SN'] > 3]

oceans_id,zspec,narrow_ha_flux,narrow_ha_luminosity,narrow_ha_SN
float64,float64,float64[3],float64[3],float64
462.0,4.1427,3.92249154227167e-17 .. 4.022380321312242e-17,6.878811097859926e+42 .. 7.053984462648885e+42,79.54211039265724
597.0,3.151,1.9200935619633158e-18 .. 3.011115716342615e-18,1.7503050595620665e+41 .. 2.744851176862613e+41,4.323806255975659
1459.0,4.9465,3.057816265215638e-17 .. 3.1450363611063254e-17,8.154915153039789e+42 .. 8.387523138588061e+42,71.10996053729775
1613.0,4.2807,2.731621676461777e-17 .. 2.816997352286723e-17,5.177710229740488e+42 .. 5.3395373648445e+42,64.97922257121178
1781.0,4.3493,2.4901277189254568e-17 .. 2.5724918340344637e-17,4.9011791093279825e+42 .. 5.0632917902408906e+42,61.44726129309155
1794.0,3.6817,1.0268657320931322e-16 .. 1.0449965040090048e-16,1.3596718302505837e+43 .. 1.3836787661763359e+43,114.25533720926398
1994.0,5.4228,2.2155555383938743e-17 .. 2.3074376228685434e-17,7.332104774142888e+42 .. 7.636177074999451e+42,49.20869112200349
2255.0,2.9669,5.241213755710133e-18 .. 6.265768060057464e-18,4.132169734062974e+41 .. 4.939927723844554e+41,11.194568180560958
2638.0,5.0198,2.8880241534620824e-17 .. 2.9874396996967106e-17,7.973279301138303e+42 .. 8.247746506010426e+42,59.09515930781744
